In [1]:
from model_builder import ModelBuilder
from model_builder_keras import KerasModelBuilder

from preprocessing import Preprocessor
from plotting_other import Plotter
from plotting import plot_dataset
#from shapley import ProcessAttributor
from shapley_improved import ProcessAttributorSHAP
from shapley_improved_other import ProcessAttributorSHAPMLP

from universal_filtering import CustomSpearmanFilter
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
#from sklearn.linear_model import Ridge
#from sklearn.linear_model import Lasso
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

# Basic Deep Learning with Sklearn MLP
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance

# Deep Learning with Keras Tensorflow
#import keras
from keras import layers, optimizers, callbacks, Sequential
#https://ipython.org/ipython-doc/3/config/extensions/autoreload.html
%load_ext autoreload
%autoreload 2

I0000 00:00:1784911333.477587 1482749 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784911333.519424 1482749 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784911334.788713 1482749 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
train_mixed_unseen_type = [
    pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260704T110043Z/datasets/chipseq_2_0607.parquet"),
    pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260701T114734Z/datasets/rnaseq_1_02027.parquet"),
    #pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260701T215234Z/datasets/sarek_1_0207.parquet"),
    pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260704T093159Z/datasets/ampliseq_2_0607.parquet")

]
#test_mixed_unseen_type = pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260704T093159Z/datasets/ampliseq_2_0607.parquet")
test_mixed_unseen_type2 = pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260701T215234Z/datasets/sarek_1_0207.parquet")


In [ ]:
training_data = pd.concat(train_mixed_unseen_type, ignore_index=True)
test_data = test_mixed_unseen_type2

features = [
    "delta_cpu_ns",
    "delta_io_bytes",
    "delta_net_send_bytes",
    "context_switches",
    "syscall_count",
    "delta_rss_memory",
    "delta_cpu_time_psutil",
    "delta_cpu_time_proc",
    "syscall_class_file",
    "syscall_class_network",
    "syscall_class_memory",
    "syscall_class_process",
    "syscall_class_other",
    "syscall_class_sched",
    "syscall_class_signal",
    "syscall_class_time",
    "delta_cycles",
    "delta_cache_misses",
    "delta_instructions",
    "delta_branch_instructions",
]


In [5]:
preprocessor_train = Preprocessor(training_data, features)
X_train_FULL, y_train, t_train, _ = preprocessor_train.preprocess_no_split()

Dropped 0 timestamps.


In [6]:
#plot_dataset(t_train, y_train, "multi_training")


In [7]:
#build_model = ExplainableBoostingRegressor( interactions=2, max_rounds=2000, n_jobs=-1, random_state=42)
model = RandomForestRegressor(n_estimators=100,  n_jobs=-1, random_state=42)
#build_model = MLPRegressor(activation="relu", solver="adam", random_state=42)



In [8]:
#These thresholds could be fine tuned
automatic_feature_selection = Pipeline(steps=[
    ('variance', VarianceThreshold(threshold=0.01)), #explain this

    ('decorrelate', CustomSpearmanFilter(threshold=0.80)),
    ('scaler', StandardScaler()),
    ('select_features', SelectFromModel(model, threshold='0.5*median'))
])

automatic_feature_selection.set_output(transform="pandas")
automatic_feature_selection.fit_transform(X_train_FULL, y_train)
good_features = automatic_feature_selection.get_feature_names_out().tolist()
X_train = X_train_FULL[good_features]
print("Selected columns:")
print(good_features)


#plot_dataset(t_train, y_train, "multi_training")

Selected columns:
['delta_cpu_ns', 'delta_io_bytes', 'delta_net_send_bytes', 'context_switches', 'syscall_count', 'delta_rss_memory', 'delta_cpu_time_proc', 'syscall_class_file', 'syscall_class_network', 'syscall_class_memory', 'syscall_class_process', 'syscall_class_other']


In [9]:
preprocessor_test = Preprocessor(test_data, good_features)
X_test, y_test, t_test , X_test_unaggregated = preprocessor_test.preprocess_no_split()

#plot_dataset(t_test, y_test, "multi_testing")

Dropped 0 timestamps.


In [10]:
plot_dataset(t_test, y_test, "multi_testing")

### Sklearn MLP

### Deep Learning with Keras

Build the 1D-CNN in Keras (Tutorials)

- https://keras.io/guides/sequential_model/
- https://keras.io/examples/timeseries/timeseries_classification_from_scratch/
- https://www.tensorflow.org/tutorials/structured_data/time_series#recurrent_neural_network
- https://keras.io/examples/timeseries/timeseries_classification_from_scratch/

Different: Add windowing
- https://www.tensorflow.org/tutorials/structured_data/time_series#2_split
- https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html 
- Pay attention to window_size in layers.Input(shape=(num_features, window_size))
- The data shape is (batch_size, window, features).

In [11]:
num_features = len(good_features)
window_size = 20 # Initial 1

In [12]:
# Models used
# Convolutional Neural Network (1D)
cnn_model = Sequential([

    layers.Input(shape=(num_features, window_size)), # (num_features, sequence_length) #Only current value
    layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    layers.BatchNormalization(),

    layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    layers.BatchNormalization(),

    #layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    #layers.BatchNormalization(),
    
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
    
])

# LSTM Model
lstm_model = Sequential([
    layers.Input(shape=(num_features, window_size)), # (num_features, sequence_length) #Only current value
    #layers.BatchNormalization(),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(64, return_sequences=True),
    layers.Flatten(),
    #layers.GlobalAveragePooling1D(),
    #layers.Dropout(0.2),
    #layers.Dense(32, activation='relu'),
    layers.Dense(1)
    
])

In [13]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 64)         │        21,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 12, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           769 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,553 (217.00 KB)

 Trainable params: 55,553 (217.00 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
builder_lstm = KerasModelBuilder(X_train, X_test, y_train, y_test, lstm_model, StandardScaler(), 
                                 window_size=window_size, train_epochs=18) # Maybe 20?
y_pred_lstm, learned_idle_power = builder_lstm.run_and_save_model()

Epoch 1/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - loss: 11365.6924 - mae: 83.4004 - val_loss: 749.4833 - val_mae: 22.0441
Epoch 2/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 388.0287 - mae: 15.8157 - val_loss: 317.6895 - val_mae: 12.9411
Epoch 3/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 135.3857 - mae: 8.5974 - val_loss: 225.9981 - val_mae: 10.9459
Epoch 4/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 122.3009 - mae: 8.1396 - val_loss: 220.1156 - val_mae: 10.6896
Epoch 5/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 116.5695 - mae: 7.9150 - val_loss: 222.5092 - val_mae: 10.3791
Epoch 6/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 113.2329 - mae: 7.7602 - val_loss: 243.2034 - val_mae: 10.7925
Epoch 7/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 110.0708 - mae: 7.6661 - val_loss: 245.2939 - val_mae: 10.9548
Epoch 8/18
388/388 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 106.8828 - mae: 7.5522 - val_loss: 242.8289 - val_mae: 10.8025
Epoc

In [15]:
y_pred_lstm.shape

(8294, 1)

In [16]:
y_test.shape

(8313,)

In [17]:
t_test.shape

(8313,)

In [18]:
#plotter = Plotter(y_pred=y_pred_lstm,y_test=y_test, t_test= t_test,alg_name="lstm",window_start=150, window_end=200)
plotter = Plotter(y_pred=y_pred_lstm,y_test=y_test[window_size - 1:], t_test= t_test[window_size - 1:],alg_name="lstm")
#plotter.plot_only("lstm_")
plotter.plot_and_save("ltsm_windowing_")

In [23]:
builder_cnn = KerasModelBuilder(X_train, X_test, y_train, y_test, cnn_model, StandardScaler(), window_size=20, 
                                train_epochs=20)
y_pred_cnn, learned_idle_power = builder_cnn.run_and_save_model()

Epoch 1/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 133.7915 - mae: 8.8192 - val_loss: 493.4059 - val_mae: 15.2920
Epoch 2/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 107.5054 - mae: 7.8764 - val_loss: 474.8928 - val_mae: 15.0915
Epoch 3/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 101.2832 - mae: 7.6750 - val_loss: 474.3715 - val_mae: 14.8407
Epoch 4/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 96.0187 - mae: 7.4319 - val_loss: 458.2207 - val_mae: 14.9401
Epoch 5/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 95.0592 - mae: 7.3833 - val_loss: 446.5529 - val_mae: 14.5001
Epoch 6/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 89.9500 - mae: 7.1881 - val_loss: 529.4622 - val_mae: 16.3683
Epoch 7/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 88.6799 - mae: 7.1405 - val_loss: 413.2819 - val_mae: 14.0376
Epoch 8/20
388/388 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 86.8689 - mae: 7.0392 - val_loss: 410.7949 - val_mae: 13.9832
Epoch 9/20
388/388 ━━

In [ ]:
plotter = Plotter(y_pred=y_pred_cnn,y_test=y_test[window_size - 1:], t_test= t_test[window_size - 1:],alg_name="cnn_1d")
#plotter.plot_only("cnn_1d")
plotter.plot_and_save("cnn_1d_windowing")

In [20]:

builder_rf = ModelBuilder(X_train, X_test, y_train, y_test, model, StandardScaler())
y_pred_rf, learned_idle_power = builder_rf.run_and_save_model()


  R² Score:  0.8262
  MAE:       8.16 Ws (4.08% of mean)
----------------------------------
The model's learned baseline idle interval energy is: 142.14 Ws
----------------------------------
/n


In [25]:
plotter = Plotter(y_pred=y_pred_rf,y_test=y_test[window_size - 1:], t_test= t_test[window_size - 1:],alg_name="rf")
#plotter.plot_only("cnn_1d")
plotter.plot_and_save("rf_")